In [13]:

from __future__ import annotations
import sys
import time
from pathlib import Path

project_dir = Path.cwd()
if str(project_dir) not in sys.path:
    sys.path.append(str(project_dir))

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from FHF_HF2 import HFLArgs, run_hfl_with_third_party_reconciliation


In [2]:

print('torch version:', torch.__version__)
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')
print('working directory:', project_dir)


torch version: 2.3.1+cu118
device: cuda
working directory: E:\yjz\FHF\UF2_modified_code\UF2


In [3]:

# -------------------------
# Parameter setup
# -------------------------
args = HFLArgs()

# Change this path to your own dataset if needed.
# Example (current folder):
args.csv_path = "E:/yjz/Datasets/AusGrid2013/AusGrid2013_univariate.csv" # str(project_dir / 'GEF2017_univarivate.csv')

# Basic experiment settings
args.model_type = 'lstm'
args.truncated = None # '2013-05-01'
args.fh = 0
args.lags = 48
args.test_ratio = 0.2
args.batch_size = 256

# FL / HFL settings
args.num_rounds = 100
args.edge_agg_rounds = 1
args.edge_agg_rounds_by_edge = None
args.edge_execution_mode = 'sequential'   # each edge finishes all its own rounds first
args.local_epochs = 1
args.client_frac_per_edge = 1.0


# Early stopping / stop criteria
args.early_stop_enabled = True
args.early_stop_tol = 1e-4
args.early_stop_metric = 'avg_normalised_loss_delta'
args.min_rounds = 20
args.edge_trainable = False
args.cloud_trainable = False

# Reconciliation settings
args.recon_method = ('bu', 'mint_cov', 'mint_shrinkage', 'mint_ols', 'mint_var', 'wls_structure')
args.validate_p2p = False
args.p2p_atol = 1e-7

# Runtime / logging
args.device = 'cuda' if torch.cuda.is_available() else 'cpu'
args.verbose_cloud_rounds = True
args.verbose_edge_rounds = False
args.print_every = 10
args.seed = 42
args.torch_num_threads = 1

args


HFLArgs(csv_path='E:/yjz/Datasets/AusGrid2013/AusGrid2013_univariate.csv', partition_col='partition_id', series_col='unique_id', time_col='ds', target_col='y', truncated=None, lags=48, fh=0, test_ratio=0.2, batch_size=256, model_type='lstm', input_size=1, hidden_size=64, num_layers=2, dropout=0.1, c_in=1, d_model=128, n_heads=4, e_layers=3, device='cuda', num_rounds=100, edge_agg_rounds=1, edge_agg_rounds_by_edge=None, edge_execution_mode='sequential', local_epochs=1, lr=0.001, weight_decay=0.0, optimizer='adam', client_frac_per_edge=1.0, edge_trainable=False, cloud_trainable=False, hier_train_mode='post_agg_local_finetune', early_stop_enabled=True, early_stop_tol=0.0001, early_stop_metric='avg_normalised_loss_delta', min_rounds=20, recon_method=('bu', 'mint_cov', 'mint_shrinkage', 'mint_ols', 'mint_var', 'wls_structure'), ridge=1e-06, td_eps=1e-08, validate_p2p=False, p2p_atol=1e-07, seed=42, torch_num_threads=1, verbose_cloud_rounds=True, verbose_edge_rounds=False, print_every=10)

In [4]:

# Direct HFL run
result = run_hfl_with_third_party_reconciliation(args)

nodes = result['nodes']
layout = result['layout']
cid2series = result['cid2series']
cloud_round_logs = result['cloud_round_logs']
edge_round_logs = result['edge_round_logs']
timings = result['timings']

print('Run finished.')
print('Number of hierarchy nodes:', len(layout.node_ids))
print('Cloud node id:', layout.cloud_id)
print('Edge ids:', layout.edge_ids)
print('Number of client nodes:', len(layout.client_ids))


[Cloud Round 10/100] avg_normalised_loss=0.455323, avg_actual_loss=0.0497977, delta=0.000466145, stopped=False
[Cloud Round 20/100] avg_normalised_loss=0.432481, avg_actual_loss=0.0468317, delta=0.0031937, stopped=False
[Cloud Round 28/100] avg_normalised_loss=0.420411, avg_actual_loss=0.0455033, delta=7.17172e-05, stopped=True
[Early Stop] normalised loss delta 7.17172e-05 < 0.0001 at round 28
Run finished.
Number of hierarchy nodes: 400
Cloud node id: 0
Edge ids: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100]
Number of client nodes: 299


In [5]:
# Timing breakdown: setup, FL training, reconciliation, and evaluation
# evaluation_sec is updated after the evaluation cell below.
def make_timing_df(timings):
    ordered_modules = [
        'setup_sec',
        'data_preparation_sec',
        'model_server_initialization_sec',
        'fl_training_sec',
        'residual_collection_sec',
        'recon_matrix_sec',
        'p2p_reconcile_sec',
        'reconciliation_sec',
        'evaluation_sec',
        'total_sec',
    ]
    return pd.DataFrame(
        [{'module': key, 'seconds': timings.get(key, np.nan)} for key in ordered_modules]
    )

# At this point evaluation has not been run, so evaluation_sec is still 0.0.
timing_df = make_timing_df(timings)
timing_df

,module,seconds
0,setup_sec,60.101620
1,data_preparation_sec,60.077431
2,model_server_initialization_sec,0.024189
3,fl_training_sec,5081.276677
4,residual_collection_sec,109.292845
5,recon_matrix_sec,1.861088
6,p2p_reconcile_sec,18.248399
7,reconciliation_sec,129.402332
8,evaluation_sec,0.000000
9,total_sec,5270.780629


In [6]:
# Round logs, including average training loss and early-stopping status
cloud_logs_df = pd.DataFrame(cloud_round_logs)
edge_logs_df = pd.DataFrame(edge_round_logs)

print('Cloud round logs:')
display(cloud_logs_df.tail())
print('Edge round logs:')
display(edge_logs_df.head())

print('Early stop summary:')
display(pd.DataFrame([result.get('early_stop', {})]))

Cloud round logs:


,cloud_round,num_cloud_rounds,edge_agg_rounds_by_edge,edge_upload_weights,cloud_upload_weight,edge_execution_mode,hier_train_mode,avg_normalised_loss,avg_actual_loss,normalised_loss_delta,...,early_stop_tol,early_stop_enabled,stopped,num_train_nodes,node_normalised_losses,node_actual_losses,node_train_losses,cloud_normalised_loss,cloud_actual_loss,cloud_train_loss
23,24,100,"{1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: ...","{1: 13968, 2: 13968, 3: 27936, 4: 13968, 5: 13...",4176432,sequential,post_agg_local_finetune,0.423375,0.045723,0.002359,...,0.0001,True,False,299,"{101: 0.6761570970558766, 102: 0.8579073264972...","{101: 0.02086996272829107, 102: 0.070476208177...","{101: 0.6761570970558766, 102: 0.8579073264972...",NaN,NaN,NaN
24,25,100,"{1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: ...","{1: 13968, 2: 13968, 3: 27936, 4: 13968, 5: 13...",4176432,sequential,post_agg_local_finetune,0.422857,0.045677,0.000518,...,0.0001,True,False,299,"{101: 0.678007734668487, 102: 0.85879345509604...","{101: 0.02092708366355088, 102: 0.070549003512...","{101: 0.678007734668487, 102: 0.85879345509604...",NaN,NaN,NaN
25,26,100,"{1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: ...","{1: 13968, 2: 13968, 3: 27936, 4: 13968, 5: 13...",4176432,sequential,post_agg_local_finetune,0.420856,0.045415,0.002001,...,0.0001,True,False,299,"{101: 0.6840949077649897, 102: 0.8529260528592...","{101: 0.021114967685635637, 102: 0.07006700283...","{101: 0.6840949077649897, 102: 0.8529260528592...",NaN,NaN,NaN
26,27,100,"{1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: ...","{1: 13968, 2: 13968, 3: 27936, 4: 13968, 5: 13...",4176432,sequential,post_agg_local_finetune,0.420339,0.045483,0.000517,...,0.0001,True,False,299,"{101: 0.6786537565043429, 102: 0.8593745740964...","{101: 0.020947023503761677, 102: 0.07059674218...","{101: 0.6786537565043429, 102: 0.8593745740964...",NaN,NaN,NaN
27,28,100,"{1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: ...","{1: 13968, 2: 13968, 3: 27936, 4: 13968, 5: 13...",4176432,sequential,post_agg_local_finetune,0.420411,0.045503,0.000072,...,0.0001,True,True,299,"{101: 0.6736269949637738, 102: 0.8558492689198...","{101: 0.020791869682994825, 102: 0.07030714133...","{101: 0.6736269949637738, 102: 0.8558492689198...",NaN,NaN,NaN


Edge round logs:


,cloud_round,num_cloud_rounds,edge_id,edge_round,num_edge_rounds_for_edge,edge_upload_weight,edge_execution_mode,hier_train_mode,avg_normalised_loss,avg_actual_loss,avg_train_loss,num_train_events,selected_client_ids
0,1,100,1,1,1,13968,sequential,post_agg_local_finetune,0.793174,0.024482,0.793174,1,[101]
1,1,100,2,1,1,13968,sequential,post_agg_local_finetune,0.931711,0.076539,0.931711,1,[102]
2,1,100,3,1,1,27936,sequential,post_agg_local_finetune,0.459364,0.047970,0.459364,2,"[103, 104]"
3,1,100,4,1,1,13968,sequential,post_agg_local_finetune,0.587940,0.021541,0.587940,1,[105]
4,1,100,5,1,1,13968,sequential,post_agg_local_finetune,0.532479,0.056484,0.532479,1,[106]


Early stop summary:


,stopped_early,stop_reason,stopped_round,final_avg_normalised_loss,final_avg_actual_loss,final_normalised_loss_delta,final_avg_train_loss,final_early_stop_value,early_stop_metric,early_stop_tol,min_rounds
0,True,abs(avg_normalised_loss - previous_avg_normali...,28,0.420411,0.045503,0.000072,0.420411,0.000072,avg_normalised_loss_delta,0.0001,20


In [7]:
# Last few cloud rounds show convergence and whether early stopping was triggered.
cloud_logs_df.tail()

,cloud_round,num_cloud_rounds,edge_agg_rounds_by_edge,edge_upload_weights,cloud_upload_weight,edge_execution_mode,hier_train_mode,avg_normalised_loss,avg_actual_loss,normalised_loss_delta,...,early_stop_tol,early_stop_enabled,stopped,num_train_nodes,node_normalised_losses,node_actual_losses,node_train_losses,cloud_normalised_loss,cloud_actual_loss,cloud_train_loss
23,24,100,"{1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: ...","{1: 13968, 2: 13968, 3: 27936, 4: 13968, 5: 13...",4176432,sequential,post_agg_local_finetune,0.423375,0.045723,0.002359,...,0.0001,True,False,299,"{101: 0.6761570970558766, 102: 0.8579073264972...","{101: 0.02086996272829107, 102: 0.070476208177...","{101: 0.6761570970558766, 102: 0.8579073264972...",NaN,NaN,NaN
24,25,100,"{1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: ...","{1: 13968, 2: 13968, 3: 27936, 4: 13968, 5: 13...",4176432,sequential,post_agg_local_finetune,0.422857,0.045677,0.000518,...,0.0001,True,False,299,"{101: 0.678007734668487, 102: 0.85879345509604...","{101: 0.02092708366355088, 102: 0.070549003512...","{101: 0.678007734668487, 102: 0.85879345509604...",NaN,NaN,NaN
25,26,100,"{1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: ...","{1: 13968, 2: 13968, 3: 27936, 4: 13968, 5: 13...",4176432,sequential,post_agg_local_finetune,0.420856,0.045415,0.002001,...,0.0001,True,False,299,"{101: 0.6840949077649897, 102: 0.8529260528592...","{101: 0.021114967685635637, 102: 0.07006700283...","{101: 0.6840949077649897, 102: 0.8529260528592...",NaN,NaN,NaN
26,27,100,"{1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: ...","{1: 13968, 2: 13968, 3: 27936, 4: 13968, 5: 13...",4176432,sequential,post_agg_local_finetune,0.420339,0.045483,0.000517,...,0.0001,True,False,299,"{101: 0.6786537565043429, 102: 0.8593745740964...","{101: 0.020947023503761677, 102: 0.07059674218...","{101: 0.6786537565043429, 102: 0.8593745740964...",NaN,NaN,NaN
27,28,100,"{1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: ...","{1: 13968, 2: 13968, 3: 27936, 4: 13968, 5: 13...",4176432,sequential,post_agg_local_finetune,0.420411,0.045503,0.000072,...,0.0001,True,True,299,"{101: 0.6736269949637738, 102: 0.8558492689198...","{101: 0.020791869682994825, 102: 0.07030714133...","{101: 0.6736269949637738, 102: 0.8558492689198...",NaN,NaN,NaN


In [8]:

# Quick inspection of saved forecasts
node0 = layout.node_ids[0]
print('example node id:', node0)
print('series name:', cid2series[node0])
print('methods:', list(nodes[node0].reconciled_forecasts.keys()))
for method, arr in nodes[node0].reconciled_forecasts.items():
    print(method, np.asarray(arr).shape)


example node id: 0
series name: GC
methods: ['bu', 'mint_cov', 'mint_shrinkage', 'mint_ols', 'mint_var', 'wls_structure']
bu (3504, 1)
mint_cov (3504, 1)
mint_shrinkage (3504, 1)
mint_ols (3504, 1)
mint_var (3504, 1)
wls_structure (3504, 1)



## Rebuild forecast tables for evaluation

This section mirrors the idea in `FHF_fh1.ipynb`, but uses the outputs already stored in the HFL nodes.


In [9]:

forecast_horizon = ['y'] + [f'post_{i}' for i in range(1, args.fh + 1)]
node_order = list(layout.node_ids)


def split_node_df(df_one, args):
    sub = df_one.sort_values('ds').copy().reset_index(drop=True)

    n_windows = len(sub)
    n_raw = n_windows + int(args.lags) + int(args.fh)
    n_train_raw = int((1 - float(args.test_ratio)) * n_raw)
    n_train_windows = n_train_raw - int(args.lags) - int(args.fh)

    if n_train_windows <= 0 or n_train_windows >= n_windows:
        raise ValueError(
            f'Bad split for node: n_windows={n_windows}, n_train_windows={n_train_windows}'
        )

    tr = sub.iloc[:n_train_windows].copy()
    ts = sub.iloc[n_train_windows:].copy()
    return tr, ts


hfl_recon_rr = {}
train_actuals_by_h = {}

for H_IDX, target_col in enumerate(forecast_horizon):
    train_frames = []
    test_frames = []

    for nid in node_order:
        sub = nodes[nid].df.copy()
        tr_df, ts_df = split_node_df(sub, args)

        ts_out = ts_df[['unique_id', 'ds', target_col]].copy()
        ts_out.columns = ['unique_id', 'ds', 'y']
        ts_out['base'] = np.asarray(nodes[nid].base_forecast)[:, H_IDX]

        for method, arr in nodes[nid].reconciled_forecasts.items():
            ts_out[method] = np.asarray(arr)[:, H_IDX]

        test_frames.append(ts_out)

        tr_out = tr_df[['unique_id', 'ds', target_col]].copy()
        tr_out.columns = ['unique_id', 'ds', 'y']
        train_frames.append(tr_out)

    hfl_recon_rr[H_IDX] = pd.concat(test_frames, ignore_index=True)
    train_actuals_by_h[H_IDX] = pd.concat(train_frames, ignore_index=True)

hfl_recon_rr[0].head()


,unique_id,ds,y,base,bu,mint_cov,mint_shrinkage,mint_ols,mint_var,wls_structure
0,GC,2013-04-19 00:00:00,78.031,78.629590,78.985123,75.624953,76.009595,78.628322,78.705293,78.628322
1,GC,2013-04-19 00:30:00,61.409,77.361415,75.703678,74.090746,73.451383,77.335562,75.781673,77.335562
2,GC,2013-04-19 01:00:00,57.268,67.890840,65.025367,65.223558,65.391775,67.856368,65.500773,67.856368
3,GC,2013-04-19 01:30:00,57.180,63.434174,61.398790,56.483580,56.910149,63.404144,61.688005,63.404144
4,GC,2013-04-19 02:00:00,51.572,62.053064,60.572320,57.177199,57.944226,62.035157,60.867523,62.035157


## Evaluation: level-wise MAE, MASE, and RMSSE

This section evaluates each method by hierarchy level. The MASE and RMSSE denominators are computed from each node's training-set true values (`node.y2`), not from the test set.

In [10]:
import time


def _as_2d(arr):
    arr = np.asarray(arr, dtype=np.float64)
    if arr.ndim == 1:
        arr = arr[:, None]
    return arr


def _target_cols(args):
    return ['y'] + [f'post_{i}' for i in range(1, int(args.fh) + 1)]


def split_node_df(df_one, args):
    time_col = getattr(args, 'time_col', 'ds')
    sub = df_one.sort_values(time_col).copy().reset_index(drop=True)

    n_windows = len(sub)
    n_raw = n_windows + int(args.lags) + int(args.fh)
    n_train_raw = int((1 - float(args.test_ratio)) * n_raw)
    n_train_windows = n_train_raw - int(args.lags) - int(args.fh)

    if n_train_windows <= 0 or n_train_windows >= n_windows:
        raise ValueError(
            f'Bad split for node: n_windows={n_windows}, n_train_windows={n_train_windows}'
        )

    tr = sub.iloc[:n_train_windows].copy()
    ts = sub.iloc[n_train_windows:].copy()
    return tr, ts


def make_naive_forecast(node, args):
    """Simple random-walk naive forecast: use the latest available lag, lags_1."""
    _, ts_df = split_node_df(node.df, args)
    y_true = _as_2d(node.test_true)
    if 'lags_1' not in ts_df.columns:
        raise KeyError("Cannot compute naive forecast because 'lags_1' is missing.")
    naive_1d = ts_df['lags_1'].to_numpy(dtype=np.float64)
    if len(naive_1d) != y_true.shape[0]:
        raise ValueError(
            f'Naive length mismatch: naive={len(naive_1d)}, y_true={y_true.shape[0]}'
        )
    return np.repeat(naive_1d[:, None], y_true.shape[1], axis=1)


def _safe_scale_denominators(train_true, eps=1e-12):
    train_true = _as_2d(train_true)
    if train_true.shape[0] <= 1:
        return np.nan, np.nan

    diff = np.diff(train_true, axis=0)
    mase_denom = np.nanmean(np.abs(diff))
    rmsse_denom = np.nanmean(diff ** 2)

    if (not np.isfinite(mase_denom)) or mase_denom <= eps:
        mase_denom = np.nan
    if (not np.isfinite(rmsse_denom)) or rmsse_denom <= eps:
        rmsse_denom = np.nan

    return float(mase_denom), float(rmsse_denom)


def compute_metrics_by_level(nodes, layout, args, include_naive=True, eps=1e-12):
    rows = []
    role_order = {'cloud': 1, 'edge': 2, 'client': 3}

    for nid in layout.node_ids:
        node = nodes[nid]
        role = layout.role_by_id[nid]
        unique_id = str(node.series_name)
        level = int(unique_id.count('/') + 1)

        y_true = _as_2d(node.test_true)
        train_true = _as_2d(node.y2)
        mase_denom, rmsse_denom = _safe_scale_denominators(train_true, eps=eps)

        method_forecasts = {'base': node.base_forecast}
        if include_naive:
            method_forecasts['naive'] = make_naive_forecast(node, args)
        method_forecasts.update(node.reconciled_forecasts)

        for method, forecast in method_forecasts.items():
            y_pred = _as_2d(forecast)
            if y_pred.shape != y_true.shape:
                raise ValueError(
                    f'Shape mismatch for node={nid}, method={method}: '
                    f'y_true={y_true.shape}, y_pred={y_pred.shape}'
                )

            error = y_true - y_pred
            mae = float(np.nanmean(np.abs(error)))
            mse = float(np.nanmean(error ** 2))
            mase = mae / mase_denom if np.isfinite(mase_denom) else np.nan
            rmsse = np.sqrt(mse / rmsse_denom) if np.isfinite(rmsse_denom) else np.nan

            rows.append({
                'node_id': int(nid),
                'unique_id': unique_id,
                'level': level,
                'role': role,
                'role_order': role_order.get(role, level),
                'method': str(method),
                'MAE': mae,
                'MASE': float(mase) if np.isfinite(mase) else np.nan,
                'RMSSE': float(rmsse) if np.isfinite(rmsse) else np.nan,
            })

    per_node_metrics = pd.DataFrame(rows)

    metrics_by_level = (
        per_node_metrics
        .groupby(['level', 'role', 'role_order', 'method'], as_index=False)
        .agg(
            MAE=('MAE', 'mean'),
            MASE=('MASE', 'mean'),
            RMSSE=('RMSSE', 'mean'),
            n_nodes=('node_id', 'nunique'),
        )
        .sort_values(['role_order', 'level', 'method'])
        .drop(columns=['role_order'])
        .reset_index(drop=True)
    )

    overall_metrics = (
        per_node_metrics
        .groupby(['method'], as_index=False)
        .agg(
            MAE=('MAE', 'mean'),
            MASE=('MASE', 'mean'),
            RMSSE=('RMSSE', 'mean'),
            n_nodes=('node_id', 'nunique'),
        )
        .sort_values(['method'])
        .reset_index(drop=True)
    )

    return per_node_metrics, metrics_by_level, overall_metrics


def build_forecast_table_hfl(nodes, layout, args, include_naive=True):
    """Build a per-series forecast table with base, reconciled, and naive forecasts."""
    rows = []
    target_cols = _target_cols(args)
    time_col = getattr(args, 'time_col', 'ds')

    for nid in layout.node_ids:
        node = nodes[nid]
        _, ts_df = split_node_df(node.df, args)
        y_true = _as_2d(node.test_true)
        base = _as_2d(node.base_forecast)
        naive = make_naive_forecast(node, args) if include_naive else None
        rec = {method: _as_2d(arr) for method, arr in node.reconciled_forecasts.items()}

        for h_idx, target_col in enumerate(target_cols):
            for t_idx, (_, row) in enumerate(ts_df.iterrows()):
                out = {
                    'node_id': int(nid),
                    'unique_id': str(node.series_name),
                    'role': layout.role_by_id[nid],
                    'ds': row[time_col],
                    'horizon_index': int(h_idx),
                    'target_col': target_col,
                    'y_true': float(y_true[t_idx, h_idx]),
                    'base': float(base[t_idx, h_idx]),
                }
                if include_naive:
                    out['naive'] = float(naive[t_idx, h_idx])
                for method, arr in rec.items():
                    out[str(method)] = float(arr[t_idx, h_idx])
                rows.append(out)

    return pd.DataFrame(rows)


def save_hfl_outputs(output_prefix, nodes, layout, args, per_node_metrics,
                     metrics_by_level, overall_metrics, cloud_round_logs,
                     edge_round_logs, timings):
    """Save forecasts, per-series metrics, aggregate metrics, round logs, and time costs."""
    forecast_table = build_forecast_table_hfl(nodes, layout, args, include_naive=True)
    timing_df = make_timing_df(timings)

    paths = {
        'forecasts': f'{output_prefix}_forecasts.csv',
        'per_node_metrics': f'{output_prefix}_per_node_metrics.csv',
        'metrics_by_level': f'{output_prefix}_metrics_by_level.csv',
        'overall_metrics': f'{output_prefix}_overall_metrics.csv',
        'cloud_round_logs': f'{output_prefix}_cloud_round_logs.csv',
        'edge_round_logs': f'{output_prefix}_edge_round_logs.csv',
        'timing': f'{output_prefix}_timing.csv',
    }

    forecast_table.to_csv(paths['forecasts'], index=False)
    per_node_metrics.to_csv(paths['per_node_metrics'], index=False)
    metrics_by_level.to_csv(paths['metrics_by_level'], index=False)
    overall_metrics.to_csv(paths['overall_metrics'], index=False)
    pd.DataFrame(cloud_round_logs).to_csv(paths['cloud_round_logs'], index=False)
    pd.DataFrame(edge_round_logs).to_csv(paths['edge_round_logs'], index=False)
    timing_df.to_csv(paths['timing'], index=False)

    return paths, forecast_table, timing_df


_eval_t0 = time.perf_counter()
per_node_metrics, metrics_by_level, overall_metrics = compute_metrics_by_level(
    nodes, layout, args, include_naive=True
)
timings['evaluation_sec'] = time.perf_counter() - _eval_t0
timings['total_sec'] = (
    timings.get('setup_sec', 0.0)
    + timings.get('fl_training_sec', 0.0)
    + timings.get('reconciliation_sec', 0.0)
    + timings.get('evaluation_sec', 0.0)
)

hfl_output_tag = 'trainable' if (bool(args.edge_trainable) or bool(args.cloud_trainable)) else 'nontrainable'
output_prefix = f'fh{args.fh + 1}_V2HF2_{hfl_output_tag}'
output_paths, forecast_table, timing_df = save_hfl_outputs(
    output_prefix=output_prefix,
    nodes=nodes,
    layout=layout,
    args=args,
    per_node_metrics=per_node_metrics,
    metrics_by_level=metrics_by_level,
    overall_metrics=overall_metrics,
    cloud_round_logs=cloud_round_logs,
    edge_round_logs=edge_round_logs,
    timings=timings,
)

result['timings'] = timings
result['per_node_metrics'] = per_node_metrics
result['metrics_by_level'] = metrics_by_level
result['overall_metrics'] = overall_metrics
result['forecast_table'] = forecast_table
result['output_paths'] = output_paths

print('Saved outputs:')
for name, path in output_paths.items():
    print(f'  {name}: {path}')

metrics_by_level


Saved outputs:
  forecasts: fh1_V2HF2_nontrainable_forecasts.csv
  per_node_metrics: fh1_V2HF2_nontrainable_per_node_metrics.csv
  metrics_by_level: fh1_V2HF2_nontrainable_metrics_by_level.csv
  overall_metrics: fh1_V2HF2_nontrainable_overall_metrics.csv
  cloud_round_logs: fh1_V2HF2_nontrainable_cloud_round_logs.csv
  edge_round_logs: fh1_V2HF2_nontrainable_edge_round_logs.csv
  timing: fh1_V2HF2_nontrainable_timing.csv


,level,role,method,MAE,MASE,RMSSE,n_nodes
0,1,cloud,base,7.975363,1.301746,1.222282,1
1,1,cloud,bu,7.899250,1.289323,1.209262,1
2,1,cloud,mint_cov,6.818162,1.112867,1.006514,1
3,1,cloud,mint_ols,7.968174,1.300573,1.221373,1
4,1,cloud,mint_shrinkage,6.780248,1.106679,1.006876,1
5,1,cloud,mint_var,7.773117,1.268736,1.192062,1
6,1,cloud,naive,7.763219,1.267120,1.198684,1
7,1,cloud,wls_structure,7.968174,1.300573,1.221373,1
8,2,edge,base,0.253587,1.130595,1.024213,100
9,2,edge,bu,0.251863,1.126376,1.022130,100


In [11]:
# Optional: overall average across all hierarchy nodes
overall_metrics

,method,MAE,MASE,RMSSE,n_nodes
0,base,0.181567,1.147780,1.017689,400
1,bu,0.180945,1.146694,1.017136,400
2,mint_cov,0.181161,1.173896,1.015712,400
3,mint_ols,0.182301,1.156943,1.019956,400
4,mint_shrinkage,0.179336,1.160358,1.011652,400
5,mint_var,0.180842,1.147915,1.016858,400
6,naive,0.185562,1.167698,1.133205,400
7,wls_structure,0.182301,1.156943,1.019956,400


In [12]:
# Final timing table, saved by save_hfl_outputs(...).
timing_df


,module,seconds
0,setup_sec,60.101620
1,data_preparation_sec,60.077431
2,model_server_initialization_sec,0.024189
3,fl_training_sec,5081.276677
4,residual_collection_sec,109.292845
5,recon_matrix_sec,1.861088
6,p2p_reconcile_sec,18.248399
7,reconciliation_sec,129.402332
8,evaluation_sec,8.357379
9,total_sec,5279.138008
